# Wisconsin Branch Review and Integration

## Objective
This notebook documents the frozen Wisconsin branch without modifying it. The purpose is to understand what can be reused, what assumptions it already makes, and how it should be integrated into the synthetic pairing workflow.


In [1]:
from __future__ import annotations

import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
plt.style.use('seaborn-v0_8-whitegrid')

CWD = Path.cwd().resolve()
if (CWD / 'src').exists() and (CWD / 'data').exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / 'src').exists() and (CWD.parent / 'data').exists():
    PROJECT_ROOT = CWD.parent
elif (CWD.parent.parent / 'src').exists() and (CWD.parent.parent / 'data').exists():
    PROJECT_ROOT = CWD.parent.parent
else:
    raise RuntimeError(f'Could not resolve dissertation_project root from {CWD}')

REPO_ROOT = PROJECT_ROOT.parent
OUTPUTS = PROJECT_ROOT / 'outputs_v2'
FIGURES = OUTPUTS / 'figures'
METRICS = OUTPUTS / 'metrics'
REPORTS = OUTPUTS / 'reports'
MODELS = PROJECT_ROOT / 'models'
DATA_ROOT = PROJECT_ROOT / 'data' / 'dataset_cancer_v1' / 'dataset_cancer_v1'
WISCONSIN_ROOT = PROJECT_ROOT / 'notebook_Wisconsin'

for path in [FIGURES, METRICS, REPORTS]:
    path.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Outputs:', OUTPUTS)

from IPython.display import display

wisconsin_df = pd.read_csv(WISCONSIN_ROOT / 'brca.csv').drop(columns=['Unnamed: 0'], errors='ignore')
wisconsin_df['label'] = wisconsin_df['y'].map({'B': 'benign', 'M': 'malignant'})
wisconsin_df.head()


Project root: /Users/sergeysotskiy/Documents/UNI/year 3/Dissertation/dissertation_project
Outputs: /Users/sergeysotskiy/Documents/UNI/year 3/Dissertation/dissertation_project/outputs_v2


,x.radius_mean,x.texture_mean,x.perimeter_mean,x.area_mean,x.smoothness_mean,x.compactness_mean,x.concavity_mean,x.concave_pts_mean,x.symmetry_mean,x.fractal_dim_mean,...,x.perimeter_worst,x.area_worst,x.smoothness_worst,x.compactness_worst,x.concavity_worst,x.concave_pts_worst,x.symmetry_worst,x.fractal_dim_worst,y,label
0,13.540,14.36,87.46,566.3,0.09779,0.08129,0.06664,0.047810,0.1885,0.05766,...,99.70,711.2,0.14400,0.17730,0.23900,0.12880,0.2977,0.07259,B,benign
1,13.080,15.71,85.63,520.0,0.10750,0.12700,0.04568,0.031100,0.1967,0.06811,...,96.09,630.5,0.13120,0.27760,0.18900,0.07283,0.3184,0.08183,B,benign
2,9.504,12.44,60.34,273.9,0.10240,0.06492,0.02956,0.020760,0.1815,0.06905,...,65.13,314.9,0.13240,0.11480,0.08867,0.06227,0.2450,0.07773,B,benign
3,13.030,18.42,82.61,523.8,0.08983,0.03766,0.02562,0.029230,0.1467,0.05863,...,84.46,545.9,0.09701,0.04619,0.04833,0.05013,0.1987,0.06169,B,benign
4,8.196,16.84,51.71,201.9,0.08600,0.05943,0.01588,0.005917,0.1769,0.06503,...,57.26,242.2,0.12970,0.13570,0.06880,0.02564,0.3105,0.07409,B,benign


In [2]:
label_counts = wisconsin_df['label'].value_counts().rename_axis('label').reset_index(name='count')
feature_summary = wisconsin_df.drop(columns=['y', 'label']).describe().T[['mean', 'std', 'min', 'max']].head(10)
label_counts.to_csv(REPORTS / 'wisconsin_label_counts.csv', index=False)
feature_summary.to_csv(REPORTS / 'wisconsin_feature_summary_head.csv')
display(label_counts)
display(feature_summary)


,label,count
0,benign,357
1,malignant,212


,mean,std,min,max
x.radius_mean,14.127292,3.524049,6.98100,28.11000
x.texture_mean,19.289649,4.301036,9.71000,39.28000
x.perimeter_mean,91.969033,24.298981,43.79000,188.50000
x.area_mean,654.889104,351.914129,143.50000,2501.00000
x.smoothness_mean,0.096360,0.014064,0.05263,0.16340
x.compactness_mean,0.104341,0.052813,0.01938,0.34540
x.concavity_mean,0.088799,0.079720,0.00000,0.42680
x.concave_pts_mean,0.048919,0.038803,0.00000,0.20120
x.symmetry_mean,0.181162,0.027414,0.10600,0.30400
x.fractal_dim_mean,0.062798,0.007060,0.04996,0.09744


In [3]:
import json
with open(WISCONSIN_ROOT / 'BreaScope AI.ipynb', 'r', encoding='utf-8') as f:
    wisconsin_nb = json.load(f)

headings = []
for cell in wisconsin_nb['cells']:
    if cell.get('cell_type') == 'markdown':
        for line in ''.join(cell.get('source', [])).splitlines():
            if line.startswith('#'):
                headings.append(line.strip())
headings_df = pd.DataFrame({'heading': headings})
headings_df.head(20)


,heading
0,# BreaScope AI: Bayesian Deep Learning for Bre...
1,## 1. Import Required Libraries
2,### Reflection
3,## 2. Load Dataset
4,### Reflection
5,### Class Label Encoding
6,### Reflection
7,### Preparing Features and Target Variables
8,### Reflection
9,## 3. Exploratory Data Analysis (EDA)


In [4]:
published_metrics = pd.DataFrame(
    [
        {'source': 'Published Wisconsin notebook', 'metric': 'accuracy', 'value': 0.96},
        {'source': 'Published Wisconsin notebook', 'metric': 'roc_auc', 'value': 0.997},
    ]
)
published_metrics.to_csv(REPORTS / 'wisconsin_published_metrics.csv', index=False)
published_metrics


,source,metric,value
0,Published Wisconsin notebook,accuracy,0.960
1,Published Wisconsin notebook,roc_auc,0.997


In [5]:
integration_contract = pd.DataFrame(
    [
        {'item': 'feature_schema', 'detail': '30 numeric diagnostic features after dropping the CSV export index column.'},
        {'item': 'label_mapping', 'detail': 'B -> benign, M -> malignant'},
        {'item': 'saved_model', 'detail': str(WISCONSIN_ROOT / 'model.pt')},
        {'item': 'saved_scaler', 'detail': str(WISCONSIN_ROOT / 'scaler.joblib')},
        {'item': 'uncertainty_method', 'detail': 'Monte-Carlo Dropout reported in the frozen notebook'},
    ]
)
integration_contract.to_csv(REPORTS / 'wisconsin_integration_contract.csv', index=False)
integration_contract


,item,detail
0,feature_schema,30 numeric diagnostic features after dropping ...
1,label_mapping,"B -> benign, M -> malignant"
2,saved_model,/Users/sergeysotskiy/Documents/UNI/year 3/Diss...
3,saved_scaler,/Users/sergeysotskiy/Documents/UNI/year 3/Diss...
4,uncertainty_method,Monte-Carlo Dropout reported in the frozen not...


## Interpretation

The Wisconsin branch is not retrained here. Instead, it is treated as a fixed monomodel with a documented dataset, artifact schema, and published reported metrics. That is enough for controlled comparison and synthetic pairing, while respecting the user's instruction not to change the published work.
